In [ ]:
%pip install "tensorflow[cuda]" mlflow pandas azure-storage-blob matplotlib opencv-python scikit-learn

In [ ]:
import os
import sys
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Num GPUs Available:", len(gpus))
if gpus:
    for gpu in gpus:
        print("  ->", gpu)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Memory growth setting:", e)
else:
    print("WARNING: No GPU detected! Check nvidia-smi and restart kernel.")

In [ ]:
import os
from azure.storage.blob import BlobServiceClient


service = BlobServiceClient.from_connection_string(os.environ["AZURE_STORAGE_CONNECTION_STRING"])
container = service.get_container_client("kcrowly-vast")

In [ ]:
import os
import tarfile

data_archive = "neurovision_data.tar.gz"
data_dir = "Data/raw" if os.path.exists("Data/raw") else ("../Data/raw" if os.path.exists("../Data/raw") else "Data/raw")

if not os.path.exists(data_dir):
    print("Data folder not found locally. Downloading from Azure Blob Storage...")
    with open(data_archive, "wb") as f:
        download_stream = container.download_blob("data/neurovision_data.tar.gz")
        f.write(download_stream.readall())
    print("Download complete. Extracting dataset and helpers...")
    with tarfile.open(data_archive, "r:gz") as tar:
        tar.extractall(".")
    data_dir = "Data/raw" if os.path.exists("Data/raw") else "../Data/raw"
    print(f"Dataset ready at: {data_dir}")
else:
    print(f"Dataset already available at: {data_dir}")

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import cv2

sys.path.insert(0, os.path.abspath("src"))
sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("."))

from neurovision import data

paths, labels, _ = data.list_image_files(data_dir)
print(f"Total MRI images: {len(paths)} | Classes: {data.CLASSES}")

plt.figure(figsize=(10, 2))
for i in range(min(10, len(paths))):
    plt.subplot(1, 10, i + 1)
    img = cv2.cvtColor(data.load_image(paths[i]), cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(data.CLASSES[labels[i]], fontsize=8)
    plt.axis("off")
plt.show()

## ResNet50 Preprocessing

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

augmentation = data.make_augmentation(rotation=0.03, zoom=0.1, translation=0.1, seed=42)

datasets = data.make_datasets(
    data_dir,
    size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    augmentation=augmentation,
    preprocess="resnet50"
)

def to_one_hot(ds):
    return ds.map(lambda x, y: (x, tf.one_hot(y, data.NUM_CLASSES)),
                  num_parallel_calls=tf.data.AUTOTUNE)

train_ds = to_one_hot(datasets["train"])
val_ds = to_one_hot(datasets["val"])
test_ds = to_one_hot(datasets["test"])

In [ ]:
from tensorflow.keras.applications import ResNet50

base_model = ResNet50(weights="imagenet",
                      include_top=False,
                      input_shape=(224, 224, 3))
base_model.trainable = False

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(data.NUM_CLASSES, activation="softmax")
])

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("resnet50_neurovision.keras", save_best_only=True)

history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=10,
                    callbacks=[early_stop, checkpoint])

In [ ]:
with open("resnet50_neurovision.keras", "rb") as f:
    container.upload_blob("models/resnet50_neurovision.keras", f, overwrite=True)
print("Uploaded resnet50_neurovision.keras to Azure Blob Storage!")

In [ ]:
import subprocess, time

subprocess.run(["pkill", "-f", "mlflow"])
time.sleep(3)

p = subprocess.Popen(
    ["mlflow", "server", "--host", "0.0.0.0", "--port", "5000",
     "--disable-security-middleware"],
    stdout=open("mlflow.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(15)

print("exit code:", p.poll())
print(open("mlflow.log").read()[-1500:])

In [ ]:
import os
print(os.path.abspath("resnet50_neurovision.keras"))

## Fine Tuning

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.2f}")

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [ ]:
fine_tune_epochs = 10
initial_epoch = history.epoch[-1] + 1
total_epochs = initial_epoch + fine_tune_epochs

early_stop_fine = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
checkpoint_fine = ModelCheckpoint("resnet50_neurovision_finetuned.keras", save_best_only=True)

history_fine = model.fit(train_ds,
                         epochs=total_epochs,
                         initial_epoch=initial_epoch,
                         validation_data=val_ds,
                         callbacks=[early_stop_fine, checkpoint_fine])

## Evaluation

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.2f}")

In [ ]:
finetuned_model_path = "resnet50_neurovision_finetuned.keras"
model.save(finetuned_model_path)

with open(finetuned_model_path, "rb") as f:
    container.upload_blob(f"models/{finetuned_mod"&&el_path}", f, overwrite=True)
print(f"Successfully uploaded {finetuned_model_path} to Azure Blob Storage!")